# Chemprop predictive model for opioid receptivity via pChEMBL values.

This process will be very similar to steps 1, 2, and 3 where we built a chemprop model to predict TRPV1 interactivity. Except this time we will be gathering training/testing datasets for these 4 opioid receptors:

- Mu (MOP/MOR)
- Delta (DOP/DOR)
- Kappa (KOP/KOR)
- Nociceptin/Orphanin FQ receptor (NOP/ORL-1)

Search strings will be:
- MOR
- DOR
- KOR
- Nociceptin

In [ ]:
!pip install chembl_webresource_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00


In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd
import numpy as np

In [ ]:
# Let's create a function since we will be doing this multiple times
def get_chemble_train_test_data(search_string):
  targets_api = new_client.target
  activities_api = new_client.activity
  molecule = new_client.molecule

  # Get targets of receptors
  targets = targets_api.search(search_string).only(['target_chembl_id', 'pref_name', 'organism'])
  targets_df = pd.DataFrame(targets)

  # Get all related activites
  queried_activities = []
  for id in targets_df['target_chembl_id']:
    activities = activities_api.filter(target_chembl_id=id).only([
        'molecule_chembl_id',
        'standard_value',
        'standard_type',
        'standard_units',
        'pchembl_value'
    ])
    queried_activities.append(activities)
    print(f"{id} data retrieved...")

  merged_activities = np.concatenate(queried_activities, axis=0)
  activities_df = pd.DataFrame(list(merged_activities))

  # Filter through data with pchemble value > 0
  activities_df['pchembl_value'] = activities_df['pchembl_value'].astype(float)
  activities_df['standard_value'] = activities_df['standard_value'].astype(float)
  filtered_df = activities_df.loc[(activities_df['pchembl_value'] > 0)]
  id_value_df = filtered_df.drop(['standard_type', 'standard_units', 'standard_value', 'type', 'units', 'value'], axis=1)

  # Get the SMILES property of each molecule
  all_smiles = []
  for i, id in enumerate(id_value_df['molecule_chembl_id'].unique()):
    smiles = molecule.filter(chembl_id=id).only(['molecule_chembl_id','molecule_structures'])
    try:
      smile_str = smiles[0]['molecule_structures']['canonical_smiles']
      id = smiles[0]['molecule_chembl_id']
      all_smiles.append({'molecule_chembl_id': id, 'canonical_smiles': smile_str})
      print(f"{i}/{len(id_value_df['molecule_chembl_id'].unique())} - {id} has a SMILES structure...")
    except TypeError:
      print(f"{i}/{len(id_value_df['molecule_chembl_id'].unique())} - {id} did not have a SMILES structure...")
  smiles_df = pd.DataFrame(all_smiles)

  # Merge SMILES with ID
  merged_df = pd.merge(smiles_df, id_value_df, on='molecule_chembl_id', how='inner')

  # save CSV files
  merged_df.to_csv(f"{search_string}_id.csv", index=False)
  model_data = merged_df.drop(columns=['molecule_chembl_id'], axis=0)
  model_data.to_csv(f"{search_string}_model.csv", index=False)

  return merged_df, f"{search_string}_id.csv", f"{search_string}_model.csv"

In [ ]:
mor = get_chemble_train_test_data('MOR')

Streaming output truncated to the last 5000 lines.
7545/12545 - CHEMBL1531675 has a SMILES structure...
7546/12545 - CHEMBL1586285 has a SMILES structure...
7547/12545 - CHEMBL1609119 has a SMILES structure...
7548/12545 - CHEMBL1473706 has a SMILES structure...
7549/12545 - CHEMBL1566963 has a SMILES structure...
7550/12545 - CHEMBL1513567 has a SMILES structure...
7551/12545 - CHEMBL1587149 has a SMILES structure...
7552/12545 - CHEMBL1380599 has a SMILES structure...
7553/12545 - CHEMBL1423609 has a SMILES structure...
7554/12545 - CHEMBL1522542 has a SMILES structure...
7555/12545 - CHEMBL1426979 has a SMILES structure...
7556/12545 - CHEMBL1713166 has a SMILES structure...
7557/12545 - CHEMBL1732484 has a SMILES structure...
7558/12545 - CHEMBL1587574 has a SMILES structure...
7559/12545 - CHEMBL1423927 has a SMILES structure...
7560/12545 - CHEMBL1548987 has a SMILES structure...
7561/12545 - CHEMBL1703982 has a SMILES structure...
7562/12545 - CHEMBL1902708 has a SMILES structur

In [ ]:
dor = get_chemble_train_test_data('DOR')

Streaming output truncated to the last 5000 lines.
2990/7990 - CHEMBL2334772 has a SMILES structure...
2991/7990 - CHEMBL2334773 has a SMILES structure...
2992/7990 - CHEMBL2334776 has a SMILES structure...
2993/7990 - CHEMBL2334775 has a SMILES structure...
2994/7990 - CHEMBL2334774 has a SMILES structure...
2995/7990 - CHEMBL2338745 has a SMILES structure...
2996/7990 - CHEMBL2338744 has a SMILES structure...
2997/7990 - CHEMBL2347239 has a SMILES structure...
2998/7990 - CHEMBL2347238 has a SMILES structure...
2999/7990 - CHEMBL2347237 has a SMILES structure...
3000/7990 - CHEMBL2347236 has a SMILES structure...
3001/7990 - CHEMBL2347235 has a SMILES structure...
3002/7990 - CHEMBL2375656 has a SMILES structure...
3003/7990 - CHEMBL2381584 has a SMILES structure...
3004/7990 - CHEMBL20226 has a SMILES structure...
3005/7990 - CHEMBL2387212 has a SMILES structure...
3006/7990 - CHEMBL2387211 has a SMILES structure...
3007/7990 - CHEMBL2387331 has a SMILES structure...
3008/7990 - CHE

In [ ]:
kor = get_chemble_train_test_data('KOR')

Streaming output truncated to the last 5000 lines.
6656/11656 - CHEMBL4644543 has a SMILES structure...
6657/11656 - CHEMBL4636305 has a SMILES structure...
6658/11656 - CHEMBL4645952 has a SMILES structure...
6659/11656 - CHEMBL4633020 has a SMILES structure...
6660/11656 - CHEMBL4633223 has a SMILES structure...
6661/11656 - CHEMBL4647696 has a SMILES structure...
6662/11656 - CHEMBL4638667 has a SMILES structure...
6663/11656 - CHEMBL4647353 has a SMILES structure...
6664/11656 - CHEMBL4649820 has a SMILES structure...
6665/11656 - CHEMBL4644691 has a SMILES structure...
6666/11656 - CHEMBL4633907 has a SMILES structure...
6667/11656 - CHEMBL4636159 has a SMILES structure...
6668/11656 - CHEMBL4640139 has a SMILES structure...
6669/11656 - CHEMBL4649485 has a SMILES structure...
6670/11656 - CHEMBL4634021 has a SMILES structure...
6671/11656 - CHEMBL4641271 has a SMILES structure...
6672/11656 - CHEMBL4648920 has a SMILES structure...
6673/11656 - CHEMBL4639005 has a SMILES structur

In [ ]:
nociceptin = get_chemble_train_test_data('Nociceptin')

CHEMBL2014 data retrieved...
CHEMBL3621 data retrieved...
CHEMBL4503 data retrieved...
CHEMBL5492 data retrieved...
CHEMBL3038507 data retrieved...
0/2872 - CHEMBL108417 has a SMILES structure...
1/2872 - CHEMBL322717 has a SMILES structure...
2/2872 - CHEMBL110138 has a SMILES structure...
3/2872 - CHEMBL110379 has a SMILES structure...
4/2872 - CHEMBL448155 has a SMILES structure...
5/2872 - CHEMBL321462 has a SMILES structure...
6/2872 - CHEMBL320061 has a SMILES structure...
7/2872 - CHEMBL107288 has a SMILES structure...
8/2872 - CHEMBL108893 has a SMILES structure...
9/2872 - CHEMBL327118 has a SMILES structure...
10/2872 - CHEMBL108904 has a SMILES structure...
11/2872 - CHEMBL324895 has a SMILES structure...
12/2872 - CHEMBL317479 has a SMILES structure...
13/2872 - CHEMBL320804 has a SMILES structure...
14/2872 - CHEMBL110933 has a SMILES structure...
15/2872 - CHEMBL106084 has a SMILES structure...
16/2872 - CHEMBL106190 has a SMILES structure...
17/2872 - CHEMBL316961 has a 

In [ ]:
nociceptin[0]

,molecule_chembl_id,canonical_smiles,pchembl_value
0,CHEMBL108417,CC(c1ccccc1)N1CC[C@H]1[C@@H](N)c1cccc(Cl)c1,5.97
1,CHEMBL322717,CCCCCCC(c1ccccc1)N1CC[C@H]1[C@@H](N)c1cccc(Cl)c1,5.99
2,CHEMBL110138,Cc1ccc(C(c2ccc(C)cc2)N2CC[C@H]2[C@H](N)c2cccc(...,5.95
3,CHEMBL110379,CCCCC(c1cccc(F)c1)N1CC[C@H]1[C@H](N)c1cccc(Cl)c1,7.58
4,CHEMBL448155,COc1ccc([C@H](N)[C@@H]2CCN2C(c2ccccc2)c2ccccc2...,6.70
...,...,...,...
3807,CHEMBL70,CN1CC[C@]23c4c5ccc(O)c4O[C@H]2[C@@H](O)C=C[C@H...,7.86
3808,CHEMBL2403213,COc1ccc2[nH]cc([C@H](CNC(=O)[C@H](CCSC)NC(=O)[...,5.78
3809,CHEMBL2403213,COc1ccc2[nH]cc([C@H](CNC(=O)[C@H](CCSC)NC(=O)[...,6.26
3810,CHEMBL2403214,CC(=O)N[C@@H](CCCNC(=N)N)C(=O)N[C@@H](Cc1ccc(O...,6.50


In [ ]:
assert False

In [ ]:
from google.colab import files

In [ ]:
# You will have to enable multiple downloads on your browser...
files.download(mor[1])
files.download(mor[2])

files.download(dor[1])
files.download(dor[2])

files.download(kor[1])
files.download(kor[2])

files.download(nociceptin[1])
files.download(nociceptin[2])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>